# OLR Quicklook (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-06-24  
**Last Modified:** 2026-06-24  
**Status:** In Progress  
**Keywords:** AOS, open loop reproduction, wavefront, Zernikes, DOF, v-modes  

## Description

Quicklook diagnostics for the Open Loop Reproduction (OLR) pipeline outputs in
`rubin-work/olr`. Reads the two per-night parquet products and compares the
open-loop reproduction against the closed-loop (measured) wavefront.

Key functionality:
1. Read the per-night nightly AOS table and the OLR parquet for one `day_obs`.
2. Compare open-loop (reproduced) vs closed-loop (measured) Zernikes and the
   wavefront RMS the AOS loop removed, with PSF FWHM overlaid.
3. Show the applied AOS correction (the trim added back to make the OLR), split
   by component: camera & M2 piston, decenter, tip/tilt, M1M3 / M2 bending
   modes, and the state v-modes — all vs seq, with block boundaries marked.

Note on the OLR at the first image: the trim added back is the full accumulated
MTAOS `aggregatedDoF`. At the start of a night/block this is often zero (loop
not yet closed), so there the OLR equals the measured wavefront; OLR and
measured diverge only once the loop applies corrections.

**Output:** Inline diagnostic plots (no files written).

**Based on:** `rubin-work/olr` pipeline; Craig Lage's Open_Loop_Reproduction
notebook; canonical `nightly_report_ts_version.ipynb` (DM-54406).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-24 | Aaron Roodman | Initial version |
| 2026-06-24 | Aaron Roodman | Mirror modes as per-mode strip charts (was heatmaps); drop tight_layout (constrained layout is on) |
| 2026-06-24 | Aaron Roodman | Add state v-mode timeline; mark block boundaries on timelines |
| 2026-06-24 | Aaron Roodman | Add first-image sanity-check section (Z5/Z6 OLR-vs-measured overlap) |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs = 20260415                 # night to inspect
base    = f"output/{day_obs}"      # pipeline output dir (run from rubin-work/olr)

# Which wavefront to compare: "deviation" (controlled quantity) or "opd"
# (the full optical path difference, used for the measured-intrinsic / MIW work).
field = "deviation"

# Zernike panels to show (label, Noll index).
zernike_panels = [("Z4 defocus", 4), ("Z5 astig", 5), ("Z6 astig", 6),
                  ("Z7 coma", 7), ("Z8 coma", 8), ("Z11 spherical", 11)]

# Strip charts: drop modes that are flat-zero all night (inactive DOF / v-modes).
drop_inactive_modes = True

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()   # note: enables figure.constrained_layout (no tight_layout needed)

<a id='functions'></a>
## Helper Functions

In [ ]:
CORNER_NAMES = ["R00", "R04", "R40", "R44"]   # 4 corner WFS sensors
NOLL = np.arange(4, 27)                        # 23 terms; array index i <-> Noll i+4

# --- 50-DOF layout (OFC aggregatedDoF order) ---
# 0-4   M2 hexapod  : dz, dx, dy, rx, ry
# 5-9   Cam hexapod : dz, dx, dy, rx, ry
# 10-29 M1M3 bending modes (20)
# 30-49 M2   bending modes (20)
DZ_COLS      = [0, 5]                  # piston            (microns)
DXDY_COLS    = [1, 2, 6, 7]            # decenter          (microns)
TIPTILT_COLS = [3, 4, 8, 9]            # tip/tilt          (arcsec)
M1M3_COLS    = list(range(10, 30))
M2BEND_COLS  = list(range(30, 50))


def stack(df, prefix):
    """Per-corner column set {prefix}_R00.. -> (n_visits, 4, 23) array."""
    return np.stack([np.vstack(df[f"{prefix}_{c}"].values) for c in CORNER_NAMES], axis=1)


def term(arr, noll):
    """Pick a single Noll term from a (..., 23) array."""
    return arr[..., noll - 4]


def vec_by_seq(src, col, seq, n):
    """Pull a per-seq vector column from src (DataFrame) aligned to `seq`.

    Returns (len(seq), n) with NaN rows where the seq is missing.
    """
    s = src.drop_duplicates("seq").set_index("seq")[col]
    out = np.full((len(seq), n), np.nan)
    for i, q in enumerate(seq):
        v = s.get(q)
        if v is not None and np.ndim(v) == 1:
            out[i] = np.asarray(v, dtype=float)[:n]
    return out


def mark_blocks(ax, seq, block):
    """Faint dashed vline at each block change (skips NaN/None block labels)."""
    b = np.asarray(block, dtype=object)
    for i in range(1, len(b)):
        if b[i] != b[i - 1] and b[i] is not None and b[i - 1] is not None:
            ax.axvline(seq[i], color="0.7", lw=0.8, ls="--", zorder=0)


def plot_dof_lines(ax, seq, dof, cols, labels, ylabel, title):
    """Line plot of selected DOF columns vs seq."""
    for col, lab in zip(cols, labels):
        ax.plot(seq, dof[:, col], ".-", ms=4, lw=0.8, label=lab)
    ax.axhline(0, lw=0.6, color="k", alpha=0.4)
    ax.set_xlabel("seq")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)


def plot_strips(ax, seq, data, labels, ylabel, drop_inactive=True, atol=1e-9):
    """Overlaid strip chart: one trace per column of `data` (n_visits, n_modes)."""
    n = 0
    for j, lab in enumerate(labels):
        y = data[:, j]
        if drop_inactive and np.nanmax(np.abs(y)) < atol:
            continue
        ax.plot(seq, y, "-", lw=1.0, label=lab)
        n += 1
    ax.axhline(0, lw=0.6, color="k", alpha=0.4)
    ax.set_xlabel("seq")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if n:
        ax.legend(fontsize=8, ncol=2, loc="best")
    return n

<a id='data'></a>
## Data Access

In [ ]:
olr_df = pd.read_parquet(f"{base}/olr.parquet")
night  = pd.read_parquet(f"{base}/nightly_aos_table.parquet")

print(f"olr.parquet            : {olr_df.shape[0]} rows, {olr_df.shape[1]} cols")
print(f"nightly_aos_table      : {night.shape[0]} rows, {night.shape[1]} cols")
print("bands:", sorted(olr_df["band"].dropna().unique().tolist()))

<a id='analysis'></a>
## Analysis

In [ ]:
olr_df = olr_df.sort_values("seq").reset_index(drop=True)
seq = olr_df["seq"].values

# Mean over the 4 corner sensors -> (n_visits, 23).
open_wf   = stack(olr_df, f"olr_{field}").mean(axis=1)    # open-loop reproduction
closed_wf = stack(olr_df, f"meas_{field}").mean(axis=1)   # measured (closed-loop)

# Wavefront RMS in quadrature over Noll 4..26.
rms_open   = np.sqrt(np.nansum(open_wf**2,   axis=1))
rms_closed = np.sqrt(np.nansum(closed_wf**2, axis=1))

# Full 50-DOF state (the correction; trim = dof_state[active 22]).
dof = np.vstack(olr_df["dof_state"].values)

# block label and state v-modes per seq.  Prefer the OLR table (newer runs carry
# them); fall back to the nightly table for older OLR parquets.
block_src  = olr_df if "block"  in olr_df.columns else night
vmode_src  = olr_df if "vmodes" in olr_df.columns else night
block  = np.array([block_src.drop_duplicates("seq").set_index("seq")["block"].get(q)
                   for q in seq], dtype=object)
vmodes = vec_by_seq(vmode_src, "vmodes", seq, 12)

# PSF FWHM from the nightly table, aligned to the OLR seqs.
fwhm = night.set_index("seq")["psf_fwhm"].reindex(seq).values

print(f"field = {field!r};  {len(seq)} visits;  dof {dof.shape};  vmodes {vmodes.shape}")
print("first seq:", int(seq[0]), "| max|trim| =", round(float(np.nanmax(np.abs(np.vstack(olr_df['trim'].values)[0]))), 3),
      "(0 -> OLR == measured there)")

<a id='results'></a>
## Results & Plots

### First-image sanity check

Where the accumulated trim is ~0 (start of night before the loop closes, or
later when the correction does not project onto a given Zernike), the OLR must
equal the measured wavefront. This zooms the first `n_zoom` seqs for Z5/Z6 and
reports where OLR first departs from measured — a quick check that the trim is
being added back correctly.

In [ ]:
n_zoom = 50
m = seq <= seq[0] + n_zoom

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ax, (label, n) in zip(axes, [("Z5 astig", 5), ("Z6 astig", 6)]):
    mark_blocks(ax, seq[m], block[m])
    ax.plot(seq[m], term(closed_wf, n)[m], "o-", ms=5, color="tab:blue",   label="closed-loop (measured)")
    ax.plot(seq[m], term(open_wf,   n)[m], "x-", ms=6, color="tab:orange", label="open-loop reproduction")
    ax.axhline(0, lw=0.6, color="k", alpha=0.4)
    ax.set_ylabel(f"{label}\n[microns]")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)
axes[1].set_xlabel("seq")
fig.suptitle(f"First-image sanity check ({field}) — {day_obs}")

# Report: max|trim| per seq, the first-image overlap, and where OLR departs from
# measured in Z6 (the loop's astigmatism correction kicking in).
trim_max = np.array([np.nanmax(np.abs(np.asarray(t, float))) for t in olr_df["trim"].values])
d6 = np.abs(term(open_wf, 6) - term(closed_wf, 6))
i0 = int(np.argmax(d6 > 0.05)) if np.any(d6 > 0.05) else -1
print(f"first seq = {int(seq[0])}, max|trim| = {trim_max[0]:.2f}  ->  "
      f"OLR==measured (Z6): {np.allclose(term(open_wf, 6)[0], term(closed_wf, 6)[0])}")
if i0 >= 0:
    print(f"OLR departs from measured (|dZ6|>0.05um) first at seq {int(seq[i0])}, "
          f"max|trim| there = {trim_max[i0]:.1f}")
print(f"OLR Z6 range (open-loop disturbance): {np.nanmin(term(open_wf, 6)):.2f} .. {np.nanmax(term(open_wf, 6)):.2f}")
print(f"measured Z6 range (closed-loop):      {np.nanmin(term(closed_wf, 6)):.2f} .. {np.nanmax(term(closed_wf, 6)):.2f}")

### Open- vs closed-loop Zernikes across the night
(dashed grey lines mark block changes)

In [ ]:
fig, axes = plt.subplots(len(zernike_panels), 1, figsize=(11, 12), sharex=True)
for ax, (label, n) in zip(axes, zernike_panels):
    mark_blocks(ax, seq, block)
    ax.plot(seq, term(closed_wf, n), ".", ms=4, color="tab:blue",   label="closed-loop (measured)")
    ax.plot(seq, term(open_wf,   n), ".", ms=4, color="tab:orange", label="open-loop reproduction")
    ax.axhline(0, lw=0.6, color="k", alpha=0.4)
    ax.set_ylabel(f"{label}\n[microns]")
    ax.grid(alpha=0.3)
axes[0].legend(ncol=2, fontsize=9, loc="upper right")
axes[-1].set_xlabel("seq")
fig.suptitle(f"OLR vs measured Zernikes ({field}) — {day_obs}")

### Wavefront RMS the loop removed (with PSF FWHM)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(seq, rms_open,   ".", ms=5, color="tab:orange", label="open-loop RMS WF")
ax.plot(seq, rms_closed, ".", ms=5, color="tab:blue",   label="closed-loop RMS WF")
ax.set_xlabel("seq")
ax.set_ylabel("WF RMS  [microns]")
ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.plot(seq, fwhm, "-", lw=1, color="gray", alpha=0.7)
ax2.set_ylabel("PSF FWHM [arcsec]", color="gray")
ax.legend(loc="upper left")
ax.set_title(f"Wavefront RMS: closed vs open loop ({field}) — {day_obs}")

### Applied AOS correction (trim), split by component

The trim added back to make the OLR, from `dof_state`. Hexapod piston/decenter
are in microns, tip/tilt in arcsec; bending modes (microns) and the state
v-modes are shown as per-mode strip charts.

In [ ]:
# Camera & M2 piston (dz)
fig, ax = plt.subplots(figsize=(11, 4))
plot_dof_lines(ax, seq, dof, DZ_COLS, ["M2 dz", "Cam dz"],
               "piston [microns]", f"Camera & M2 piston (dz) — {day_obs}")

In [ ]:
# Camera & M2 decenter (dx, dy)
fig, ax = plt.subplots(figsize=(11, 4))
plot_dof_lines(ax, seq, dof, DXDY_COLS, ["M2 dx", "M2 dy", "Cam dx", "Cam dy"],
               "decenter [microns]", f"Camera & M2 decenter (dx, dy) — {day_obs}")

In [ ]:
# Camera & M2 tip/tilt (rx, ry)
fig, ax = plt.subplots(figsize=(11, 4))
plot_dof_lines(ax, seq, dof, TIPTILT_COLS, ["M2 rx", "M2 ry", "Cam rx", "Cam ry"],
               "tip/tilt [arcsec]", f"Camera & M2 tip/tilt (rx, ry) — {day_obs}")

In [ ]:
# M1M3 bending modes (per-mode strip chart)
fig, ax = plt.subplots(figsize=(11, 5))
labels = [f"M1M3 b{k}" for k in range(1, len(M1M3_COLS) + 1)]
mark_blocks(ax, seq, block)
n = plot_strips(ax, seq, dof[:, M1M3_COLS], labels, "amplitude [microns]", drop_inactive_modes)
ax.set_title(f"M1M3 bending modes ({n} shown) — {day_obs}")

In [ ]:
# M2 bending modes (per-mode strip chart)
fig, ax = plt.subplots(figsize=(11, 5))
labels = [f"M2 b{k}" for k in range(1, len(M2BEND_COLS) + 1)]
mark_blocks(ax, seq, block)
n = plot_strips(ax, seq, dof[:, M2BEND_COLS], labels, "amplitude [microns]", drop_inactive_modes)
ax.set_title(f"M2 bending modes ({n} shown) — {day_obs}")

### State v-mode timeline

The 12 OFC v-modes computed from `dof_state` (`StateEstimator.get_vmodes_from_dofs`).
These are the applied correction expressed in v-mode space — useful to sanity-check
that the controlled modes behave reasonably across the night.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
labels = [f"v{j+1}" for j in range(vmodes.shape[1])]
mark_blocks(ax, seq, block)
n = plot_strips(ax, seq, vmodes, labels, "v-mode amplitude", drop_inactive_modes)
ax.set_title(f"State v-modes (from dof_state) — {day_obs}")